# Fase 11 — Marco de apoyo a decisiones para anchoveta

Esta fase transforma los 97 hallazgos validados y sus evaluaciones de calidad y transferibilidad en productos auditables para deliberación experta.

Principios:

- los conteos de hallazgos no se interpretan como tamaños de efecto;
- una recomendación, propuesta o escenario no demuestra implementación;
- implementación no equivale a efectividad evaluada;
- ninguna salida se interpreta como consejo automático de manejo;
- las opciones se clasifican como candidatas para revisión, prueba piloto, MSE, monitoreo, investigación o uso contextual;
- toda recomendación candidata requiere revisión experta antes de comunicación externa o adopción.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from evidence_review.decision_support import (
    build_anchoveta_recommendation_framework,
    build_climate_response_management_chains,
    build_decision_support_summary,
    build_management_option_portfolio,
    build_operational_readiness_matrix,
    build_research_priority_matrix,
    initialise_recommendation_review,
    load_decision_support_config,
    read_csv_robust,
    split_recommendation_review,
    validate_decision_support_outputs,
)

CONFIG_PATH = ROOT / "config" / "decision_support.yml"
config = load_decision_support_config(CONFIG_PATH, project_root=ROOT)
paths = config["paths"]

INTERIM = ROOT / "data" / "interim"
INTERIM.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Configuration: {CONFIG_PATH.relative_to(ROOT)}")

## 1. Cargar las salidas validadas de las fases 09–10

La entrada principal es `evidence_synthesis_matrix.csv`. Los resúmenes y matrices complementarias se cargan para auditoría y control de cobertura.

In [ ]:
input_keys = [
    "synthesis_matrix_csv",
    "synthesis_theme_summary_csv",
    "management_option_summary_csv",
    "evidence_gap_matrix_csv",
    "recommendation_candidates_csv",
    "transferability_validated_csv",
]

loaded = {}
for key in input_keys:
    path = ROOT / paths[key]
    if not path.exists():
        raise FileNotFoundError(path)
    loaded[key], encoding = read_csv_robust(path)
    print(f"{path.relative_to(ROOT)}: {len(loaded[key])} [{encoding}]")

matrix = loaded["synthesis_matrix_csv"]
gaps = loaded["evidence_gap_matrix_csv"]

print(f"Validated synthesis findings: {len(matrix)}")
print(f"Sources represented: {matrix['source_id'].nunique()}")
print(f"Gap records: {len(gaps)}")

## 2. Construir cadenas, portafolio, madurez y prioridades

Las cadenas se conservan a nivel de hallazgo. El portafolio se construye únicamente con medidas de manejo explícitamente codificadas; no se inventan opciones ausentes.

In [ ]:
chains = build_climate_response_management_chains(matrix, config)
portfolio = build_management_option_portfolio(matrix, config)
readiness = build_operational_readiness_matrix(portfolio)
priorities = build_research_priority_matrix(gaps, matrix, config)

print(f"Evidence chains: {len(chains)}")
print(f"Explicit management options: {len(portfolio)}")
print(f"Operational-readiness rows: {len(readiness)}")
print(f"Research priorities: {len(priorities)}")

display(
    chains.groupby("chain_class", dropna=False)
    .size()
    .rename("findings")
    .reset_index()
)
display(
    portfolio[
        [
            "management_measure",
            "findings",
            "sources",
            "readiness_class",
            "decision_pathway",
            "evaluated_implementation",
        ]
    ]
)
display(priorities.head(30))

## 3. Generar el marco condicional de recomendaciones y preservar revisión experta

Cada fila es una **recomendación candidata**, no una instrucción automática. Las decisiones humanas previas se preservan por `recommendation_id`.

In [ ]:
candidates = build_anchoveta_recommendation_framework(portfolio, config)
review_path = ROOT / paths["recommendation_review_working_csv"]

existing_review = pd.DataFrame()
if review_path.exists():
    existing_review, review_encoding = read_csv_robust(review_path)
else:
    review_encoding = "not_loaded"

framework = initialise_recommendation_review(candidates, existing_review)
framework.to_csv(review_path, index=False, encoding="utf-8-sig")

print(f"Recommendation candidates: {len(framework)}")
print(f"Existing expert-review rows: {len(existing_review)} [{review_encoding}]")
print(f"Saved: {review_path.relative_to(ROOT)}")
print(framework["expert_review_status"].value_counts(dropna=False))

display(
    framework[
        [
            "recommendation_id",
            "management_measure",
            "decision_pathway",
            "readiness_class",
            "candidate_statement",
            "expert_review_status",
        ]
    ]
)

## 4. Validar trazabilidad, puntajes y lenguaje condicional

La validación impide clasificar una opción como lista para apoyo a decisiones sin implementación evaluada y detecta afirmaciones incondicionales de efectividad o adopción inmediata.

In [ ]:
issues = validate_decision_support_outputs(
    chains,
    portfolio,
    readiness,
    priorities,
    framework,
    config,
)

issues_path = ROOT / paths["issues_csv"]
issues.to_csv(issues_path, index=False, encoding="utf-8-sig")

print(f"Validation issues: {len(issues)}")
display(issues.head(100))

## 5. Exportar productos de apoyo a decisiones

Las recomendaciones validadas contienen únicamente filas con `expert_review_status = accepted` o `corrected`. Las pendientes permanecen separadas.

In [ ]:
# Recarga defensiva para preservar ediciones hechas fuera del kernel.
framework, _ = read_csv_robust(review_path)
review_groups = split_recommendation_review(framework)
summary = build_decision_support_summary(
    chains,
    portfolio,
    readiness,
    priorities,
    framework,
)

outputs = {
    paths["chains_csv"]: chains,
    paths["management_portfolio_csv"]: portfolio,
    paths["operational_readiness_csv"]: readiness,
    paths["research_priority_csv"]: priorities,
    paths["recommendation_framework_csv"]: review_groups["all"],
    paths["recommendation_validated_csv"]: review_groups["validated"],
    paths["recommendation_pending_csv"]: review_groups["pending"],
    paths["recommendation_rejected_csv"]: review_groups["rejected"],
    paths["decision_summary_csv"]: summary,
}

for relative_path, frame in outputs.items():
    path = ROOT / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"{path.relative_to(ROOT)}: {len(frame)}")

display(summary)
display(readiness)
display(review_groups["validated"])

## Criterio para avanzar a la fase 12

La generación de tablas, figuras y narrativa final puede comenzar cuando:

1. `Validation issues = 0`;
2. las 97 cadenas mantienen trazabilidad a hallazgo y fuente;
3. todas las medidas del portafolio corresponden a `management_measures` explícitas;
4. la madurez operacional distingue propuesta, implementación y efectividad evaluada;
5. todas las recomendaciones candidatas fueron revisadas por expertos;
6. `anchoveta_recommendation_framework_pending_review.csv` contiene 0 filas;
7. las recomendaciones aceptadas o corregidas conservan requisitos, adaptación y caveats;
8. no se comunica una opción como política adoptada ni como efectiva sin evidencia correspondiente.

La siguiente fase será:

```text
notebooks/12_build_final_evidence_products.ipynb
```